# WebLooper - Lyrics Processor (Google Colab)

**This notebook runs the heavy AI (lyrics transcription + timing) using your free Colab GPU and writes results back to your Google Drive.**

It integrates perfectly with weblooper's existing Drive sync.

### Quick Steps
1. Runtime → Change runtime type → GPU (T4 recommended, free).
2. Run all cells (it will ask for Drive permission).
3. In the config cell, the `SESSION_FOLDER_ID` is usually already filled (when you clicked 'Run in my Colab' from weblooper it uploaded a ready copy with the ID baked in). If empty, paste the folder ID that weblooper shows you.
4. Cell 1 mounts Drive and installs dependencies (pip download cache is stored on Drive so subsequent runs are much faster \u2014 ~1-2 min vs ~5 min). On first run it **automatically restarts the runtime**. This is expected! After restart, re-run from Cell 2 onward (or just \"Run all\" again \u2014 Cell 1 will detect packages are already installed and skip).
5. In the config cell choose your model (USE_PARAKEET recommended as the best for singing; exactly one must be True). The notebook will find your `vocals.webm`, run the chosen model, and write `lyricTrack.json` + patch `meta.json`.
6. Go back to weblooper and click 'Load results from Colab/Drive' (or reload) — lyrics appear with proper timing!

In [ ]:
# @title 1. Install dependencies (cached to Drive for fast re-runs)
import subprocess, sys, os

# --- Mount Drive early so we can use it as a pip download cache ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Pip cache on Drive: downloaded .whl files persist across sessions.
# First run: downloads + installs (~5 min). Subsequent runs: skips downloads,
# just unpacks from cached wheels (~1-2 min). Never breaks across Colab updates
# because packages are still installed locally each time.
PIP_CACHE = '/content/drive/MyDrive/.weblooper_colab_cache/pip'
os.makedirs(PIP_CACHE, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = PIP_CACHE

def _is_installed():
    """Check if key packages are importable (skip reinstall on re-run after restart)."""
    try:
        import nemo.collections.asr
        import pydub
        import numpy as np
        # Verify numpy is actually usable (the _center import that fails with stale numpy)
        from numpy._core.umath import _center  # noqa: F401
        return True
    except (ImportError, AttributeError):
        return False

# Skip the entire install if packages are already working (post-restart re-run)
if _is_installed():
    print('Dependencies already installed (post-restart). Skipping to next cell.')
else:
    print(f'Pip cache dir: {PIP_CACHE}')
    # Check if cache has content (indicates a repeat run that will be faster)
    cache_files = os.listdir(PIP_CACHE) if os.path.isdir(PIP_CACHE) else []
    if cache_files:
        print(f'Found cached wheels ({len(cache_files)} entries) \u2014 install will be faster (\u223c1-2 min).')
    else:
        print('No cache yet \u2014 first-time install will take \u223c5 min (wheels will be cached for next time).')

    # --- System dependencies required by NeMo / audio processing ---
    print('\nInstalling system dependencies...')
    subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'sox', 'libsndfile1', 'ffmpeg'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('System deps: sox, libsndfile1, ffmpeg  \u2713')

    # --- Cython (build dependency for some NeMo sub-packages) ---
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'Cython'])

    # --- Upgrade numba so it accepts numpy 2.x (Colab ships numba 0.60 which caps numpy<2.1) ---
    print('Upgrading numba for numpy 2.x compatibility...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numba'])

    # --- Core ASR packages ---
    # NeMo >= 2.6.1 supports NumPy 2.x natively. Let pip resolve versions freely.
    print('Installing nemo_toolkit[asr] (this takes 2-4 minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nemo_toolkit[asr]'])

    print('Installing whisperx + utilities...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'whisperx', 'google-auth', 'google-auth-oauthlib',
                           'google-auth-httplib2', 'google-api-python-client', 'pydub'])

    # --- Ensure numpy is consistent: install the version whisperx/nemo agreed on ---
    # After all packages are installed, force numpy to a known-good 2.x version.
    # This resolves the case where Colab's pre-installed numpy 2.0.2 files linger on disk.
    print('Pinning numpy to a consistent version...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           '--force-reinstall', '--no-deps', 'numpy>=2.1,<3'])

    print('\n=== Install complete. Restarting runtime to load new packages... ===')
    print('This is normal! After restart, just re-run all cells (this cell will skip).')

    # Auto-restart the Colab runtime so Python loads the new packages cleanly.
    # os.kill is the most reliable method across Colab runtime versions.
    os.kill(os.getpid(), 9)

In [ ]:
# @title 2. Verify installation
import os

# WhisperX requires this env var on Colab to avoid pyannote weights_only errors
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = 'true'

import numpy as np
import pandas as pd

print('=== INSTALL SUMMARY ===')
print(f'numpy : {np.__version__}')
print(f'pandas: {pd.__version__}')

import nemo.collections.asr as nemo_asr
print('nemo.collections.asr: OK \u2713  (Parakeet path ready)')

try:
    import whisperx
    print(f'whisperx: OK \u2713  (alternative model \u2014 set USE_WHISPERX=True in config to use)')
except ImportError as e:
    print(f'whisperx: NOT AVAILABLE ({e})')
    print('  (Parakeet still works fine \u2014 only set USE_WHISPERX=True if you need it)')

print('=== All good ===')

In [ ]:
# @title 3. Authenticate Google Drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
import io
import json
import os
from pathlib import Path
import time

auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Authenticated to your Google Drive.')

In [ ]:
# @title 4. CONFIGURATION - Paste your session folder ID here
# weblooper will tell you the exact value when you click the button.
# When launched via the weblooper 'Run in my Colab' button, the ID below is pre-filled automatically (no paste needed).
SESSION_FOLDER_ID = "__WEBLOOPER_SESSION_FOLDER_ID__"   # replaced by weblooper on Drive upload (or paste manually)

USE_PARAKEET = True   # Best for song lyrics + speed (recommended)
USE_WHISPERX = False  # Alternative for very accurate word-level timing
# Exactly one of the two above must be True. No silent fallbacks — choose deliberately.

print('Configuration ready.')

In [ ]:
# @title 5. Locate vocal stem in the Drive folder
def list_files_in_folder(folder_id):
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType)"
    ).execute()
    return results.get('files', [])

def download_file(file_id, dest_path):
    request = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(dest_path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
    return dest_path

if not SESSION_FOLDER_ID:
    raise RuntimeError("Please paste your SESSION_FOLDER_ID above.")

files = list_files_in_folder(SESSION_FOLDER_ID)
vocal_file = next((f for f in files if 'vocal' in f['name'].lower() and f['name'].endswith(('.webm', '.mp3', '.wav'))), None)

if not vocal_file:
    raise RuntimeError("Could not find a vocals stem in the folder. Make sure the session is uploaded to Drive.")

work_dir = "/content/weblooper_lyrics"
os.makedirs(work_dir, exist_ok=True)
local_vocals = os.path.join(work_dir, vocal_file['name'])
download_file(vocal_file['id'], local_vocals)
print(f"Downloaded vocal stem: {local_vocals}")

OUTPUT_FOLDER_ID = SESSION_FOLDER_ID

In [ ]:
# @title 6. Run the AI model (Parakeet or WhisperX)
from pydub import AudioSegment

audio = AudioSegment.from_file(local_vocals)
duration_sec = len(audio) / 1000.0
print(f"Processing audio of length {duration_sec:.1f} seconds...")

# Convert to mono 16kHz WAV (models expect single-channel input;
# stereo vocals.webm causes shape mismatch errors in Parakeet)
local_vocals_mono = os.path.join(work_dir, 'vocals_mono.wav')
audio.set_channels(1).set_frame_rate(16000).export(local_vocals_mono, format='wav')
print(f"Converted to mono 16kHz WAV for model input.")

if USE_PARAKEET:
    print("Loading Parakeet TDT 0.6B (excellent for song lyrics)...")
    import nemo.collections.asr as nemo_asr
    model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")
    result = model.transcribe([local_vocals_mono], timestamps=True)[0]
    text = result.text
    chunks = []
    if hasattr(result, 'timestamp') and result.timestamp:
        for w in result.timestamp.get('word', []):
            chunks.append({"text": w['word'], "timestamp": [w['start'], w['end']]})
    else:
        words = text.split()
        for i, w in enumerate(words):
            s = (i / max(1, len(words))) * duration_sec
            e = ((i + 1) / max(1, len(words))) * duration_sec
            chunks.append({"text": w, "timestamp": [s, e]})
    print("Parakeet finished.")

elif USE_WHISPERX:
    print("Loading WhisperX (best word-level timing)...")
    import whisperx
    device = "cuda"
    model = whisperx.load_model("large-v3", device, compute_type="float16")
    audio_wav = whisperx.load_audio(local_vocals_mono)
    result = model.transcribe(audio_wav, batch_size=16)
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio_wav, device)
    text = " ".join([seg["text"] for seg in result["segments"]])
    chunks = []
    for seg in result["segments"]:
        for word in seg.get("words", []):
            chunks.append({"text": word["word"], "timestamp": [word["start"], word["end"]]})
    print("WhisperX finished.")

else:
    raise RuntimeError("Set exactly one of USE_PARAKEET=True or USE_WHISPERX=True in the CONFIG cell.")

# Clean up the large temporary WAV file (can be 50-100MB+)
os.remove(local_vocals_mono)
print("Cleaned up temporary mono WAV.")

print("=== RAW TEXT (first 400 chars) ===")
print(text[:400] + "..." if len(text) > 400 else text)

In [ ]:
# @title 7. Build LyricTrack (weblooper format) — gap-based line splitting from word timestamps
import re

# --- Gap-based line splitting ---
# Song transcription models (Parakeet, WhisperX) often return text WITHOUT
# punctuation. Splitting on punctuation alone results in one giant block.
# Instead, we detect natural phrase boundaries by looking at silence gaps
# between consecutive words in the timestamp data.

GAP_THRESHOLD = 0.35   # seconds of silence between words to trigger a new line
MAX_WORDS_PER_LINE = 12  # safety cap even if no gap detected

has_real_timestamps = len(chunks) > 0 and chunks[0].get('timestamp', [0, 0])[1] > 0

segments = []

if has_real_timestamps:
    # Build lines by detecting gaps between words
    current_line_chunks = [chunks[0]]

    for i in range(1, len(chunks)):
        prev_end = chunks[i - 1]['timestamp'][1]
        curr_start = chunks[i]['timestamp'][0]
        gap = curr_start - prev_end

        # Start a new line if there's a significant gap or we hit the word cap
        if gap >= GAP_THRESHOLD or len(current_line_chunks) >= MAX_WORDS_PER_LINE:
            # Flush current line
            line_text = ' '.join(c['text'] for c in current_line_chunks)
            start = round(current_line_chunks[0]['timestamp'][0], 3)
            end = round(current_line_chunks[-1]['timestamp'][1], 3)
            segments.append({
                "id": f"colab_{int(time.time())}_{len(segments)}",
                "start": start,
                "end": end,
                "text": line_text,
                "source": "ai",
                "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
            })
            current_line_chunks = []

        current_line_chunks.append(chunks[i])

    # Flush the last line
    if current_line_chunks:
        line_text = ' '.join(c['text'] for c in current_line_chunks)
        start = round(current_line_chunks[0]['timestamp'][0], 3)
        end = round(current_line_chunks[-1]['timestamp'][1], 3)
        segments.append({
            "id": f"colab_{int(time.time())}_{len(segments)}",
            "start": start,
            "end": end,
            "text": line_text,
            "source": "ai",
            "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
        })

    print(f"Built {len(segments)} segments using gap-based splitting (threshold={GAP_THRESHOLD}s, max_words={MAX_WORDS_PER_LINE}).")

else:
    # Fallback: uniform distribution (only if model returned no timestamps)
    def smart_split_lyrics(txt, max_chars=75):
        parts = re.split(r'([.!?\u3002\uff01\uff1f\n])', txt)
        lines = []
        current = ""
        for p in parts:
            current += p
            if len(current.strip()) > max_chars or p in '.!?\u3002\uff01\uff1f\n':
                if current.strip():
                    lines.append(current.strip())
                current = ""
        if current.strip():
            lines.append(current.strip())
        return [l for l in lines if l]

    lines = smart_split_lyrics(text)
    for i, line in enumerate(lines):
        start = round((i / max(1, len(lines))) * duration_sec, 3)
        end = round(((i + 1) / max(1, len(lines))) * duration_sec, 3)
        segments.append({
            "id": f"colab_{int(time.time())}_{i}",
            "start": start,
            "end": end,
            "text": line,
            "source": "ai",
            "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
        })
    print(f"Built {len(segments)} segments using uniform timing (no word timestamps available).")

lyric_track = {
    "id": f"lt_colab_{int(duration_sec)}",
    "stemSessionId": SESSION_FOLDER_ID,
    "duration": duration_sec,
    "segments": segments,
    "metadata": {
        "generatedAt": int(time.time() * 1000),
        "lyricsModel": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3",
        "vocalsStemUsed": True,
        "source": "google-colab-free-gpu"
    },
    "version": 1
}

local_path = "/content/lyricTrack.json"
with open(local_path, "w") as f:
    json.dump(lyric_track, f, indent=2)

print(f"\nLyricTrack: {len(segments)} segments, duration {duration_sec:.1f}s")
print("=== TIMING PREVIEW (first 8 segments) ===")
for s in segments[:8]:
    print(f"  {s['start']:6.1f}s - {s['end']:6.1f}s : {s['text'][:60]}")

In [ ]:
# @title 8. Fetch real lyrics from lyrics.ovh and correct segment text
import urllib.request
import urllib.parse

# ===========================================================================
# YouTube title parsing + smart query building
# ===========================================================================

# Common noise words found in YouTube music video titles
_NOISE_WORDS = [
    'official music video', 'official video', 'official audio', 'official lyric video',
    'lyric video', 'lyrics video', 'music video', 'with lyrics', 'w lyrics',
    'lyrics', 'lyric', 'audio', 'video',
    'official', 'officiel', 'oficial',
    'hd', 'hq', '4k', '1080p', '720p',
    'remastered', 'remaster', 'remasterizado',
    'live', 'en vivo', 'ao vivo', 'concert', 'tour',
    'full song', 'full album', 'full',
    'visualizer', 'visualiser', 'animated',
    'explicit', 'clean version', 'clean',
    'radio edit', 'single version', 'album version',
    'subtitulado', 'legendado', 'sub espanol', 'traduzione',
]

def _clean_title(raw):
    """Remove parenthesized/bracketed content and noise words from a title."""
    s = raw
    # Remove content in parentheses and brackets: (Official Video), [HD], (2017 Remaster)
    s = re.sub(r'\([^)]*\)', '', s)
    s = re.sub(r'\[[^\]]*\]', '', s)
    # Remove noise words (longest first to avoid partial matches)
    for word in sorted(_NOISE_WORDS, key=len, reverse=True):
        s = re.sub(r'\b' + re.escape(word) + r'\b', '', s, flags=re.IGNORECASE)
    # Remove trailing year like "2017" at the end
    s = re.sub(r'\b(19|20)\d{2}\b', '', s)
    # Collapse separators and extra whitespace
    s = re.sub(r'[|/]{1,2}|\u2014|\u2013|--|::', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    # Remove trailing/leading punctuation junk
    s = s.strip(' -\u2014\u2013|/')
    return s

def _parse_artist_title(raw):
    """Try to split 'Artist - Song Title' from common YouTube patterns."""
    # Try common separators: " - ", " \u2014 ", " \u2013 ", " | "
    for sep in [' - ', ' \u2014 ', ' \u2013 ', ' | ', ' // ']:
        if sep in raw:
            parts = raw.split(sep, 1)
            artist = parts[0].strip()
            title = _clean_title(parts[1])
            if artist and title:
                return artist, title
    return None, None

def build_search_queries(raw_title):
    """Build a prioritized list of search queries from a YouTube video title."""
    queries = []
    
    # Tier 1: Try to parse "Artist - Title" directly
    artist, title = _parse_artist_title(raw_title)
    if artist and title:
        queries.append(f"{artist} {title}")
        queries.append(title)  # title alone as fallback
    
    # Tier 2: Full title cleaned of noise
    cleaned = _clean_title(raw_title)
    if cleaned:
        queries.append(cleaned)
    
    # Tier 3: Raw title as last resort (sometimes works if it's already clean)
    queries.append(raw_title.strip())
    
    # Deduplicate while preserving priority order
    seen = set()
    unique = []
    for q in queries:
        q_norm = q.lower().strip()
        if q_norm and q_norm not in seen:
            seen.add(q_norm)
            unique.append(q)
    return unique

def _lyrics_ovh_suggest(query):
    """Search lyrics.ovh suggest API. Returns list of results or []."""
    url = f"https://api.lyrics.ovh/suggest/{urllib.parse.quote(query, safe='')}"
    req = urllib.request.Request(url, headers={'User-Agent': 'weblooper-colab/1.0'})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode())
    return data.get('data', [])

def _lyrics_ovh_get(artist, title):
    """Fetch lyrics from the direct /v1 endpoint. Returns lyrics string or None."""
    url = f"https://api.lyrics.ovh/v1/{urllib.parse.quote(artist, safe='')}/{urllib.parse.quote(title, safe='')}"
    req = urllib.request.Request(url, headers={'User-Agent': 'weblooper-colab/1.0'})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode())
    return data.get('lyrics', '')

# ===========================================================================
# Main lyrics lookup logic
# ===========================================================================

# --- Extract song title from meta.json in the Drive folder ---
meta_file_entry = next((f for f in files if f['name'] == 'meta.json'), None)
raw_song_title = None

if meta_file_entry:
    meta_path = os.path.join(work_dir, 'meta.json')
    download_file(meta_file_entry['id'], meta_path)
    with open(meta_path) as f:
        session_meta = json.load(f)
    # Try youtubeVideoTitle first, then fileName
    raw_song_title = session_meta.get('youtubeVideoTitle', '')
    if not raw_song_title:
        fn = session_meta.get('fileName', '')
        # Strip prefixes like "YouTube \u2014 " and file extensions
        raw_song_title = re.sub(r'^youtube\s*[\u2014\u2013-]\s*', '', fn, flags=re.IGNORECASE)
        raw_song_title = re.sub(r'\.(mp3|wav|webm|ogg|m4a)$', '', raw_song_title, flags=re.IGNORECASE)
    raw_song_title = raw_song_title.strip()

real_lyrics = None
artist_name = None
song_title = None

if not raw_song_title:
    print("Could not determine song title from meta.json. Skipping lyrics lookup.")
    print("Segments will use AI-transcribed text as-is.")
else:
    print(f"Raw title from session: {raw_song_title}")

    # --- Tier 1: Direct /v1 lookup if we can parse Artist - Title ---
    parsed_artist, parsed_title = _parse_artist_title(raw_song_title)
    if parsed_artist and parsed_title:
        print(f"  Tier 1: Trying direct lookup: {parsed_artist} / {parsed_title}")
        try:
            direct_lyrics = _lyrics_ovh_get(parsed_artist, parsed_title)
            if direct_lyrics and direct_lyrics.strip():
                real_lyrics = direct_lyrics
                artist_name = parsed_artist
                song_title = parsed_title
                print(f"  \u2713 Direct lookup succeeded!")
        except Exception as e:
            print(f"  Tier 1 failed: {e}")

    # --- Tier 2: Progressive suggest search with cleaned queries ---
    if not real_lyrics:
        queries = build_search_queries(raw_song_title)
        print(f"  Tier 2: Trying suggest search with {len(queries)} queries: {queries}")

        for i, query in enumerate(queries):
            try:
                results = _lyrics_ovh_suggest(query)
                if not results:
                    print(f"    Query {i+1} '{query}': no results")
                    continue

                # Pick best result: among those within 10s of our duration, prefer highest rank (most popular)
                best = results[0]
                if duration_sec > 0:
                    candidates = [r for r in results[:5] if abs(r.get('duration', 0) - duration_sec) < 10]
                    if candidates:
                        best = max(candidates, key=lambda r: r.get('rank', 0))
                    else:
                        # No close duration match, fall back to closest duration
                        best = min(results[:5], key=lambda r: abs(r.get('duration', 0) - duration_sec))

                artist_name = best['artist']['name']
                song_title = best['title']
                match_dur = best.get('duration', 0)
                print(f"    Query {i+1} '{query}': found {artist_name} \u2014 {song_title} ({match_dur}s vs our {duration_sec:.0f}s)")

                # Fetch the actual lyrics
                fetched = _lyrics_ovh_get(artist_name, song_title)
                if fetched and fetched.strip():
                    real_lyrics = fetched
                    print(f"  \u2713 Got lyrics via suggest search!")
                    break
                else:
                    print(f"    (lyrics endpoint returned empty for this match)")
            except Exception as e:
                print(f"    Query {i+1} '{query}': error - {e}")
                continue

    # --- Apply lyrics correction if we found real lyrics ---
    if real_lyrics and real_lyrics.strip():
        # Split into lines, filter empty lines and bracketed annotations like [guitar solo]
        real_lines = [l.strip() for l in real_lyrics.split('\n')
                      if l.strip() and not l.strip().startswith('[')]
        print(f"\nApplying {len(real_lines)} real lyric lines to {len(segments)} segments...")

        # Trigram similarity matching (same as weblooper browser-side)
        def _normalize(s):
            return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', '', s.lower())).strip()

        def _trigram_similarity(a, b):
            na, nb = _normalize(a), _normalize(b)
            if na == nb:
                return 1.0
            if not na or not nb:
                return 0.0
            ta = set(na[i:i+3] for i in range(len(na) - 2))
            tb = set(nb[i:i+3] for i in range(len(nb) - 2))
            if not ta or not tb:
                return 0.0
            intersection = len(ta & tb)
            return (2 * intersection) / (len(ta) + len(tb))

        # --- Merge short consecutive segments into phrases for better matching ---
        # Short fragments (<=3 words) that are close together (<0.5s gap) get
        # concatenated into a single phrase. After matching, the real lyric line
        # is distributed back across the original segments by word proportion.
        MERGE_MAX_WORDS = 3
        MERGE_MAX_GAP = 0.5  # seconds

        # Build groups: each group is a list of consecutive segment indices that should be matched together
        groups = []
        i = 0
        while i < len(segments):
            group = [i]
            # Keep merging the next segment if current group's last segment is short and close to the next
            while i + 1 < len(segments):
                curr_seg = segments[group[-1]]
                next_seg = segments[i + 1]
                curr_words = len(curr_seg['text'].split())
                gap = next_seg['start'] - curr_seg['end']
                if curr_words <= MERGE_MAX_WORDS and gap < MERGE_MAX_GAP:
                    i += 1
                    group.append(i)
                else:
                    break
            groups.append(group)
            i += 1

        merged_count = sum(1 for g in groups if len(g) > 1)
        if merged_count > 0:
            print(f"  Merged {merged_count} groups of short segments for better matching.")

        # For each group, build merged text, match, then distribute
        corrected = 0
        for group in groups:
            merged_text = ' '.join(segments[idx]['text'] for idx in group)

            # Find best matching real lyric line
            best_score = 0.0
            best_line = merged_text
            for real_line in real_lines:
                score = _trigram_similarity(merged_text, real_line)
                if score > best_score:
                    best_score = score
                    best_line = real_line

            if best_score > 0.3:
                if len(group) == 1:
                    # Single segment: direct replacement
                    segments[group[0]]['text'] = best_line
                    segments[group[0]]['source'] = 'lyrics.ovh'
                    corrected += 1
                else:
                    # Multiple segments: distribute words proportionally by original word counts
                    real_words = best_line.split()
                    orig_word_counts = [len(segments[idx]['text'].split()) for idx in group]
                    total_orig_words = sum(orig_word_counts)

                    # Distribute real words to each segment proportionally
                    word_offset = 0
                    for j, idx in enumerate(group):
                        if j == len(group) - 1:
                            # Last segment gets all remaining words
                            assigned = real_words[word_offset:]
                        else:
                            # Proportional share (rounded)
                            share = round(len(real_words) * orig_word_counts[j] / total_orig_words)
                            share = max(1, share)  # at least 1 word
                            assigned = real_words[word_offset:word_offset + share]
                            word_offset += share
                        segments[idx]['text'] = ' '.join(assigned) if assigned else segments[idx]['text']
                        segments[idx]['source'] = 'lyrics.ovh'
                        corrected += 1

        print(f"Corrected {corrected}/{len(segments)} segments with real lyrics (threshold > 0.3).")

        # Update lyric_track with corrected segments + metadata
        lyric_track['segments'] = segments
        lyric_track['metadata']['lyricsSource'] = f"lyrics.ovh ({artist_name} - {song_title})"

        # Re-write the local file with corrected text
        with open(local_path, "w") as f:
            json.dump(lyric_track, f, indent=2)

        print("\n=== CORRECTED PREVIEW (first 8 segments) ===")
        for s in segments[:8]:
            src_tag = ' [real]' if s.get('source') == 'lyrics.ovh' else ' [ai]'
            print(f"  {s['start']:6.1f}s - {s['end']:6.1f}s :{src_tag} {s['text'][:55]}")
    else:
        if raw_song_title:
            print("\nCould not find lyrics on lyrics.ovh. Keeping AI-transcribed text.")
            print("You can manually correct later in weblooper (Use my own lyrics button).")

In [ ]:
# @title 9. Write back to your Drive folder (lyricTrack.json + patch meta.json)
from googleapiclient.http import MediaIoBaseUpload

if OUTPUT_FOLDER_ID:
    # Upload sidecar
    media = MediaIoBaseUpload(open(local_path, "rb"), mimetype="application/json")
    drive_service.files().create(
        body={"name": "lyricTrack.json", "parents": [OUTPUT_FOLDER_ID]},
        media_body=media
    ).execute()
    print("Uploaded lyricTrack.json")

    # Patch meta.json so weblooper picks it up automatically
    try:
        meta_search = drive_service.files().list(
            q=f"'{OUTPUT_FOLDER_ID}' in parents and name='meta.json' and trashed=false",
            fields="files(id)"
        ).execute().get('files', [])
        if meta_search:
            meta_id = meta_search[0]['id']
            req = drive_service.files().get_media(fileId=meta_id)
            meta_bytes = io.BytesIO()
            downloader = MediaIoBaseDownload(meta_bytes, req)
            done = False
            while not done:
                _, done = downloader.next_chunk()
            current_meta = json.loads(meta_bytes.getvalue().decode('utf-8'))
            current_meta['lyricTrack'] = lyric_track
            media = MediaIoBaseUpload(
                io.BytesIO(json.dumps(current_meta, indent=2).encode('utf-8')),
                mimetype='application/json'
            )
            drive_service.files().update(fileId=meta_id, media_body=media).execute()
            print("Patched meta.json with lyricTrack \u2014 weblooper will see it on reload!")
    except Exception as e:
        print(f"Could not patch meta.json: {e}")

    print("\n=== SUCCESS ===")
    print("Go back to weblooper and reload this stem session. The lyrics should now have proper timing.")
else:
    print("No folder ID \u2014 please download the file manually and place it in your session folder.")
    from google.colab import files
    files.download(local_path)

**That's it!** 

The results are written back to the exact same Drive folder weblooper uses. 
Reload the session in the app and the timed lyrics will appear.